In [ ]:
import os 
import zipfile
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

from sklearn.metrics import r2_score

In [ ]:
od = pd.read_csv(r"C:\Data\PseudoPFLOW\processing\city_census_od.csv")
od= od.groupby(['city1', 'city2'])['volume'].sum().reset_index()
od

In [ ]:
zip_path = 'japan_ver85.zip'
extract_path = 'japan_shapefile'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
for file in os.listdir(extract_path):
    if file.endswith('.shp'):
        shp_file = os.path.join(extract_path, file)
        break

gdf_city = gpd.read_file(shp_file)
gdf_city

In [ ]:
city_field = 'JCODE'

df_commute = shizuoka[shizuoka['puropse'].isin([2, 3])].copy()

gdf_origin = gpd.GeoDataFrame(
    df_commute,
    geometry=gpd.points_from_xy(df_commute['origin_lon'], df_commute['origin_lat']),
    crs=gdf_city.crs
)

gdf_dest = gpd.GeoDataFrame(
    df_commute.copy(),
    geometry=gpd.points_from_xy(df_commute['destn_lon'], df_commute['destn_lat']),
    crs=gdf_city.crs
)

gdf_origin = gpd.sjoin(gdf_origin, gdf_city[[city_field, 'geometry']], how='left', predicate='within')
gdf_origin.rename(columns={city_field: 'origin_city'}, inplace=True)

gdf_dest = gpd.sjoin(gdf_dest, gdf_city[[city_field, 'geometry']], how='left', predicate='within')
gdf_dest.rename(columns={city_field: 'destn_city'}, inplace=True)

df_commute['origin_city'] = gdf_origin['origin_city'].values
df_commute['destn_city'] = gdf_dest['destn_city'].values

od_city_counts = df_commute.groupby(['origin_city', 'destn_city']).size().reset_index(name='count')

print(od_city_counts.head(10))
od_city_counts.to_csv('citylevel_commute_od.csv', index=False, encoding='utf-8-sig')

In [ ]:
od_city_counts

In [ ]:
pseudo_od = od_city_counts
pseudo_od.rename(columns={'origin_city': 'city1', 'destn_city': 'city2', 'count': 'pseudo_volume'}, inplace=True)

real_od = od
real_od.rename(columns={'volume': 'real_volume'}, inplace=True)

pseudo_od['city1'] = pseudo_od['city1'].astype(str)
pseudo_od['city2'] = pseudo_od['city2'].astype(str)

real_od['city1'] = real_od['city1'].astype(str)
real_od['city2'] = real_od['city2'].astype(str)

pseudo_od = pseudo_od[pseudo_od['city1'].str.startswith('22') & (pseudo_od['city1'].str.len() == 5)]
real_od = real_od[real_od['city1'].str.startswith('22') & (real_od['city1'].str.len() == 5)]

od_compare = pd.merge(pseudo_od, real_od, on=['city1', 'city2'], how='outer')

In [ ]:
od_compare

In [ ]:
od_compare = pd.merge(pseudo_od, real_od, on=['city1', 'city2'], how='outer')


od_compare['pseudo_volume'] = od_compare['pseudo_volume'].fillna(0)
od_compare['real_volume'] = od_compare['real_volume'].fillna(0)


od_compare['diff'] = od_compare['pseudo_volume'] - od_compare['real_volume']
od_compare['abs_diff'] = od_compare['diff'].abs()


od_compare['rel_error'] = od_compare.apply(
    lambda row: row['diff'] / row['real_volume'] if row['real_volume'] != 0 else None,
    axis=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
od_compare[od_compare['real_volume']>100]['rel_error'].dropna().clip(-2, 10).hist(bins=50)
plt.title('Relative Error Distribution (clipped at ±2)')
plt.xlabel('Relative Error')
plt.ylabel('OD Pair Count')
plt.grid(True)
plt.show()

In [ ]:
valid = od_compare[(od_compare['real_volume'] > 0) & (od_compare['pseudo_volume'] > 0)]
r2 = r2_score(valid['real_volume'], valid['pseudo_volume'])

valid = od_compare[(od_compare['real_volume'] > 0) & (od_compare['pseudo_volume'] > 0)]
r, p_value = pearsonr(valid['real_volume'], valid['pseudo_volume'])


plt.figure(figsize=(6, 6))
plt.scatter(
    od_compare['real_volume'] + 1,
    od_compare['pseudo_volume'] + 1,
    alpha=0.3
)
plt.xscale('log')
plt.yscale('log')
plt.plot([1, 1e6], [1, 1e6], 'r--')  # y = x line
plt.text(1.5, 1e6, f'Pearson r = {r:.3f}\nR² = {r2:.3f}', fontsize=12,
         bbox=dict(facecolor='white', edgecolor='gray'))
plt.xlabel('Real OD Volume (log scale)')
plt.ylabel('Pseudo OD Volume (log scale)')
plt.title('OD Pair Volume Comparison (Log-Log)')
plt.grid(True)
plt.show()

In [ ]:
od_compare[(od_compare['real_volume']>20)&(od_compare['pseudo_volume']<2)]

In [ ]:
od_compare['pseudo_volume'].sum() / od_compare['real_volume'].sum()

In [ ]:
def extract_pref_code(city_code):
    if isinstance(city_code, str):
        if len(city_code) == 5:
            return city_code[:2]
        elif len(city_code) == 4:
            return city_code[:1]
    return None

# 统一数据类型
pseudo_od['city1'] = pseudo_od['city1'].astype(str)
pseudo_od['city2'] = pseudo_od['city2'].astype(str)
real_od['city1'] = real_od['city1'].astype(str)
real_od['city2'] = real_od['city2'].astype(str)

# 提取都道府県コード
pseudo_od['to_pref'] = pseudo_od['city2'].apply(extract_pref_code)
real_od['to_pref'] = real_od['city2'].apply(extract_pref_code)

# 过滤静岡県発（city1 以 '22' 开头 且长度为5）
pseudo_shizuoka = pseudo_od[(pseudo_od['city1'].str.startswith('22')) & (pseudo_od['city1'].str.len() == 5)]
real_shizuoka = real_od[(real_od['city1'].str.startswith('22')) & (real_od['city1'].str.len() == 5)]

# 集計 by destination prefecture
pseudo_pref_dist = pseudo_shizuoka.groupby('to_pref')['pseudo_volume'].sum()
real_pref_dist = real_shizuoka.groupby('to_pref')['real_volume'].sum()

# 合并并归一化为百分比
compare_pref_dist = pd.concat([pseudo_pref_dist, real_pref_dist], axis=1, sort=True).fillna(0)
compare_pref_dist.columns = ['pseudo_volume', 'real_volume']

compare_pref_dist['pseudo_pct'] = compare_pref_dist['pseudo_volume'] / compare_pref_dist['pseudo_volume'].sum()
compare_pref_dist['real_pct'] = compare_pref_dist['real_volume'] / compare_pref_dist['real_volume'].sum()
compare_pref_dist['diff_pct'] = compare_pref_dist['pseudo_pct'] - compare_pref_dist['real_pct']
compare_pref_dist['abs_diff'] = compare_pref_dist['diff_pct'].abs()

# 百分比结果转为百分数格式便于读
compare_pref_dist_pct = compare_pref_dist.copy()
compare_pref_dist_pct[['pseudo_pct', 'real_pct', 'diff_pct']] *= 100

# 排序展示最大的差异
compare_pref_dist_pct = compare_pref_dist_pct.sort_values('abs_diff', ascending=False)


In [ ]:
compare_pref_dist_pct[['pseudo_pct', 'real_pct']].plot(kind='bar', figsize=(12,6), title='Shizuoka-Origin OD by Destination Prefecture (%)')

In [ ]:
compare_pref_dist_pct[compare_pref_dist_pct['pseudo_pct']<90][['pseudo_pct', 'real_pct']].plot(kind='bar', figsize=(12,6), title='Shizuoka-Origin OD by Destination Prefecture (%)')

In [ ]:
compare_pref_dist_pct[['pseudo_pct', 'real_pct']]

In [ ]:
# 从 pseudo_od 和 real_od 中分别筛选静岡発的记录
pseudo_shizuoka = pseudo_od[(pseudo_od['city1'].str.startswith('22')) & (pseudo_od['city1'].str.len() == 5)]
real_shizuoka = real_od[(real_od['city1'].str.startswith('22')) & (real_od['city1'].str.len() == 5)]

# 统计 pseudo 中市内 vs 市外
pseudo_within = pseudo_shizuoka[pseudo_shizuoka['city1'] == pseudo_shizuoka['city2']]['pseudo_volume'].sum()
pseudo_total = pseudo_shizuoka['pseudo_volume'].sum()
pseudo_ratio = pseudo_within / pseudo_total

# 统计 real 中市内 vs 市外
real_within = real_shizuoka[real_shizuoka['city1'] == real_shizuoka['city2']]['real_volume'].sum()
real_total = real_shizuoka['real_volume'].sum()
real_ratio = real_within / real_total

# 输出结果
print(f"【Pseudo】市内移動占比: {pseudo_ratio:.2%}（{pseudo_within:.0f}/{pseudo_total:.0f}）")
print(f"【Real】  市内移動占比: {real_ratio:.2%}（{real_within:.0f}/{real_total:.0f}）")

In [ ]:
# 假设 pseudo_od 和 real_od 已经读取
pseudo_od['city1'] = pseudo_od['city1'].astype(str)
pseudo_od['city2'] = pseudo_od['city2'].astype(str)
real_od['city1'] = real_od['city1'].astype(str)
real_od['city2'] = real_od['city2'].astype(str)

# 过滤静岡発记录
pseudo_shizuoka = pseudo_od[(pseudo_od['city1'].str.startswith('22')) & (pseudo_od['city1'].str.len() == 5)].copy()
real_shizuoka = real_od[(real_od['city1'].str.startswith('22')) & (real_od['city1'].str.len() == 5)].copy()

# 分类函数：市内 / 静岡県内の他市町村 / 県外
def classify_od(row):
    c1, c2 = row['city1'], row['city2']
    if c1 == c2:
        return '市内'
    elif c2.startswith('22') and len(c2) == 5:
        return '県内市町村間'
    else:
        return '県外'

# 添加分类列
pseudo_shizuoka['category'] = pseudo_shizuoka.apply(classify_od, axis=1)
real_shizuoka['category'] = real_shizuoka.apply(classify_od, axis=1)

# 分组汇总
pseudo_stat = pseudo_shizuoka.groupby('category')['pseudo_volume'].sum()
real_stat = real_shizuoka.groupby('category')['real_volume'].sum()

# 合并、百分比计算
df_stat = pd.concat([pseudo_stat, real_stat], axis=1).fillna(0)
df_stat.columns = ['pseudo_volume', 'real_volume']
df_stat['pseudo_pct'] = df_stat['pseudo_volume'] / df_stat['pseudo_volume'].sum() * 100
df_stat['real_pct'] = df_stat['real_volume'] / df_stat['real_volume'].sum() * 100

# 排序 & 清晰展示
df_stat = df_stat.reset_index()
df_stat = df_stat[['category', 'pseudo_volume', 'pseudo_pct', 'real_volume', 'real_pct']]
df_stat = df_stat.sort_values('category')

# 打印结果
print("📊 静岡県発 OD分類比較（市内／県内市町村間／県外）")
print(df_stat.to_string(index=False, formatters={
    'pseudo_pct': '{:.2f}%'.format,
    'real_pct': '{:.2f}%'.format
}))

In [ ]:
# 确保图表中使用英语
category_map = {
    '市内': 'Within Same City',
    '県内市町村間': 'Other Cities in Shizuoka',
    '県外': 'Outside Shizuoka'
}

df_stat['category_en'] = df_stat['category'].map(category_map)

# Bar chart: volume comparison
plt.figure(figsize=(8, 5))
bar_width = 0.35
x = range(len(df_stat))

plt.bar(x, df_stat['pseudo_volume'], width=bar_width, label='Pseudo', color='skyblue')
plt.bar([i + bar_width for i in x], df_stat['real_volume'], width=bar_width, label='Real', color='salmon')

plt.xticks([i + bar_width / 2 for i in x], df_stat['category_en'], rotation=15)
plt.ylabel('OD Volume')
plt.title('OD Type Distribution from Shizuoka (Volume)')
plt.legend()
plt.tight_layout()
plt.show()

# Pie chart: share comparison
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# Pseudo
axs[0].pie(df_stat['pseudo_pct'], labels=df_stat['category_en'], autopct='%1.1f%%', colors=['skyblue', 'lightgreen', 'orange'])
axs[0].set_title('Pseudo OD Share')

# Real
axs[1].pie(df_stat['real_pct'], labels=df_stat['category_en'], autopct='%1.1f%%', colors=['skyblue', 'lightgreen', 'orange'])
axs[1].set_title('Real OD Share')

plt.suptitle('OD Type Share from Shizuoka (Percentage)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
pseudo_od

In [ ]:
# root = './data/PseudoPFLOW/ver2.0/trip/22/'
root = "C:/Data/PseudoPFLOW/trip/22/"
shizuoka = []

for filename in os.listdir(root):
    df = pd.read_csv(root+filename, names=['pid', 'time', 'origin_lon', 'origin_lat', 'destn_lon', 'destn_lat', 'transport', 'puropse', 'occupation'])
    shizuoka.append(df)
    
shizuoka = pd.concat(shizuoka, axis=0)
shizuoka

In [ ]:
shizuoka = shizuoka[(shizuoka['puropse']==2)|(shizuoka['puropse']==3)]
shizuoka

In [ ]:
gdf_pref = gpd.read_file('prefectures.geojson')

gdf_origin = gpd.GeoDataFrame(
    shizuoka,
    geometry=gpd.points_from_xy(shizuoka['origin_lon'], shizuoka['origin_lat']),
    crs=gdf_pref.crs
)

gdf_dest = gpd.GeoDataFrame(
    shizuoka,
    geometry=gpd.points_from_xy(shizuoka['destn_lon'], shizuoka['destn_lat']),
    crs=gdf_pref.crs
)

gdf_origin = gpd.sjoin(gdf_origin, gdf_pref[['geometry', 'name']], how='left', predicate='within')
gdf_origin.rename(columns={'name': 'origin_pref'}, inplace=True)

gdf_dest = gpd.sjoin(gdf_dest, gdf_pref[['geometry', 'name']], how='left', predicate='within')
gdf_dest.rename(columns={'name': 'destn_pref'}, inplace=True)

shizuoka['origin_pref'] = gdf_origin['origin_pref']
shizuoka['destn_pref'] = gdf_dest['destn_pref']

od_pref_counts = shizuoka.groupby(['origin_pref', 'destn_pref']).size().reset_index(name='count')

In [ ]:
od_pref_counts

In [ ]:
!pip install ace-tools

In [ ]:
import ace_tools as tools; tools.display_dataframe_to_user(name="Prefecture OD分布", dataframe=od_pref_counts)

In [ ]:
shizuoka